In [1]:
import numpy as np
import faiss
import pickle
from pathlib import Path
from sentence_transformers import SentenceTransformer
from google import genai
from google.genai import types
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path=Path.cwd().parent / '.env')

BASE_DIR   = Path.cwd().parent
MODELS_DIR = BASE_DIR / "models"

c:\HireIQ\provenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
index = faiss.read_index(str(MODELS_DIR / "resume_index.faiss"))
with open(MODELS_DIR / "metadata.pkl", "rb") as f:
    metadata = pickle.load(f)

print(f"Index vectors: {index.ntotal}")   
print(f"Metadata entries: {len(metadata)}") 
print(f"Sample metadata: {metadata[0]}")

Index vectors: 2483
Metadata entries: 2483
Sample metadata: {'candidate_id': 0, 'category': 'HR', 'is_tech': False, 'text': 'hr administrator marketing associate hr administrator summary dedicated customer service manager with 15 years of experience in hospitality and customer service management respected builder and leader of customer focused teams strives to instill a shared enthusiastic commitment to customer service highlights focused on customer satisfaction team management marketing savvy conflict resolution techniques training and development skilled multi tasker client relations specialist accomplishments missouri dot supervisor training certification certified by ihg in customer loyalty and marketing by segment hilton worldwide general manager training certification accomplished trainer for cross server hospitality systems such as hilton onq micros opera pms fidelio opera reservation system ors holidex completed courses and seminars in customer service sales strategies invento

In [3]:
embedder = SentenceTransformer(str(MODELS_DIR / "all-MiniLM-L6-v2"))

def embed_text(text):
    vector = embedder.encode(text, convert_to_numpy=True)
    vector = vector / np.linalg.norm(vector)
    return vector.reshape(1, -1).astype('float32')

v = embed_text("test")
print(f"Vector shape: {v.shape}") 

Vector shape: (1, 384)


In [4]:
client = genai.Client(
    vertexai=True,
    project=os.getenv('GCP_PROJECT'),
    location=os.getenv('GCP_VERTEX_LOCATION')
)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Reply with exactly: Vertex AI working"
)
print(response.text)

Vertex AI working


In [5]:
def retrieve(query, k=5):
    q_vec = embed_text(query)
    distances, indices = index.search(q_vec, k)
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        r = metadata[idx].copy()
        r['similarity_score'] = round(float(dist), 4)
        results.append(r)
    return results

# Test retrieval
print("=== Python ML developer ===")
for r in retrieve("Python developer machine learning experience"):
    print(f"  {r['category']:<30} {r['similarity_score']:.3f}")

print("\n=== Chef kitchen ===")
for r in retrieve("head chef kitchen restaurant"):
    print(f"  {r['category']:<30} {r['similarity_score']:.3f}")

=== Python ML developer ===
  ENGINEERING                    0.485
  ENGINEERING                    0.417
  ENGINEERING                    0.399
  INFORMATION-TECHNOLOGY         0.385
  AGRICULTURE                    0.360

=== Chef kitchen ===
  CHEF                           0.670
  CHEF                           0.652
  CHEF                           0.648
  CHEF                           0.647
  CHEF                           0.642


In [6]:
def generate_answer(query, chunks):
    context_parts = []
    for i, chunk in enumerate(chunks):
        context_parts.append(
            f"Candidate {i+1} | Category: {chunk['category']}\n"
            f"{chunk['text'][:500]}"
        )
    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""You are an expert recruitment assistant.
Based ONLY on the resume excerpts below, answer the recruiter's question.
Refer to candidates as Candidate 1, 2, etc.
Do not invent skills. If information is insufficient, say so clearly.

Resumes:
{context}

Question: {query}
Answer:"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text


def rag_query(question, k=5):
    chunks  = retrieve(question, k=k)
    answer  = generate_answer(question, chunks)
    sources = [
        f"Candidate {i+1} ({c['category']})"
        for i, c in enumerate(chunks)
    ]
    return {'answer': answer, 'sources': sources}

In [7]:
def rag_query(question, k=5):
    # Step 1: semantic retrieval
    chunks = retrieve(question, k=k)
    
    # Step 2: LLM generation
    answer = generate_answer(question, chunks)
    
    # Step 3: build source list for auditability
    sources = [
        f"Candidate {i+1} ({c['category']}, score: {c['similarity_score']:.3f})"
        for i, c in enumerate(chunks)
    ]
    
    return {
        'answer':  answer,
        'sources': sources
    }

In [8]:
# Test 1 — specific skill query
r = rag_query("Which candidates have Python and machine learning experience?")
print("Q1:\n", r['answer'])
print("Sources:", r['sources'])
print("\n" + "="*60 + "\n")

# Test 2 — experience level
r = rag_query("Find candidates with more than 5 years of experience")
print("Q2:\n", r['answer'])
print("Sources:", r['sources'])
print("\n" + "="*60 + "\n")

# Test 3 — negative test (hallucination check)
r = rag_query("Find candidates with blockchain and Web3 experience")
print("Q3:\n", r['answer'])
print("Sources:", r['sources'])
print("\n" + "="*60 + "\n")

# Test 4 — cross category
r = rag_query("Find me a chef with Italian cuisine experience")
print("Q4:\n", r['answer'])
print("Sources:", r['sources'])

Q1:
 Candidate 2 has Python and machine learning experience.

Candidate 1 mentions "predictive modelling," which is a machine learning technique, but does not explicitly mention Python.
Sources: ['Candidate 1 (ENGINEERING, score: 0.422)', 'Candidate 2 (ENGINEERING, score: 0.392)', 'Candidate 3 (AGRICULTURE, score: 0.382)', 'Candidate 4 (INFORMATION-TECHNOLOGY, score: 0.379)', 'Candidate 5 (ENGINEERING, score: 0.373)']


Q2:
 Here are the candidates with more than 5 years of experience:

*   **Candidate 1:** Has 10 years of staffing, technology recruiting, and staffing experience.
*   **Candidate 5:** Has 7 years of professional IT experience.

For Candidate 2, the information provided is insufficient to determine total years of experience, as only a start date of January 2015 is mentioned without an end date or overall experience summary.
For Candidate 3 and Candidate 4, the information provided is insufficient to determine years of experience.
Sources: ['Candidate 1 (HR, score: 0.465)